# 지우 dev26＋GMM의 지도 클래스 분류: 기본Bayes판정
정상2성분/가스1성분, 원본전처리와인과피처. 다른모델없이두클래스확률비교.
고정log odds0과185안쪽선택운용점을구분하고둘다보고한다.


In [1]:
import nbformat,json
from pathlib import Path
import pandas as pd,numpy as np
from sklearn.metrics import confusion_matrix
ROOT=Path.cwd();OUT=ROOT/"reports/tables"
n=nbformat.read("notebooks/185_cn7_jiwoo_dev26_supervised_class_gmm.ipynb",as_version=4)
s="\n".join(c.source for c in n.cells if c.cell_type=="code")
exec(s[:s.index("records=[];choices=[]")])
pp=pd.read_csv(OUT/"cn7_185_supervised_class_gmm_predictions.csv",parse_dates=["TimeStamp"])
# Original normal2-component GMM family / single gas-component classifier.
# Fixed Bayes argmax: label gas iff log posterior odds>=0 (P(gas)>=.5).
# This threshold is mathematical, not optimized with evaluation labels.
model="GMM_N2_G1"
p=pp[pp.model.eq(model)].copy()
p["prediction_bayes"]=p.score.ge(0).astype(int)
m=p[p.eligible.eq(1)]
tn,fp,fn,tp=confusion_matrix(m.actual,m.prediction_bayes,labels=[0,1]).ravel()
result={"model":model,"decision":"Bayes logodds>=0, no evaluation-label cutoff fit","N":len(m),"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),"F1":float(2*tp/(2*tp+fp+fn))}
parent=np.ones(N,bool);ids=np.flatnonzero(eligible)
final=fit_models(features(parent)[ids],y[ids])
score=predict_scores(final,F[cols].to_numpy(float))[model]
F["supervised_GMM_score"]=score
F["supervised_GMM_prediction"]=(score>=0).astype(int)
result["unlabeled_clean_shots"]=len(F)
result["unlabeled_alarms"]=int(F.supervised_GMM_prediction.sum())
result["unlabeled_alarm_percent"]=float(100*F.supervised_GMM_prediction.mean())
F[["TimeStamp","run_id","supervised_GMM_score","supervised_GMM_prediction"]].to_csv(OUT/"cn7_186_supervised_GMM_bayes_unlabeled.csv",index=False)
p.to_csv(OUT/"cn7_186_supervised_GMM_bayes_labeled.csv",index=False)
p[p.actual.eq(1)].to_csv(OUT/"cn7_186_supervised_GMM_bayes_all13_gas.csv",index=False)
(OUT/"cn7_186_supervised_GMM_bayes_summary.json").write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
assert not final.get("empty_gas")
assert final["gas_count"]==13
a=pd.read_csv(OUT/"cn7_185_supervised_class_gmm_audit.csv")
assert a.timestamp_overlap.eq(0).all() and a.test_raw_in_training_history.eq(0).all()
print(json.dumps(result,ensure_ascii=False,indent=2))
print("\nALL GAS\n",p[p.actual.eq(1)][["TimeStamp","score","prediction_bayes"]].to_string(index=False))
print("\nTRAIN CLASSES normal",int((y[ids]==0).sum()),"gas",int((y[ids]==1).sum()))
lines=["# 지우dev26＋GMM 직접지도분류 최종확인","",
 "이전7개앙상블, 비지도teacher＋LR/RF는사용하지않는다.",
 "지우원본dev26/Prep/full covarianceGMM을유지하고정상라벨/가스라벨별로GMM을학습한다.",
 "정상2성분/가스1성분은원본GMM2정상분포와단순가스분포를결합한분류기한개다.공분산보정은소수라벨의수치안정을위한내부정규화이지별도탐지모델이아니다.",
 "지도GMM의표준판정은두클래스사후확률을비교하는Bayes argmax이다. log odds0은가스확률.5에해당하며평가정답에맞춰숫자를선택하지않았다.",
 "185의안쪽F1선택기준도숨기지않고별도로기록한다.해당엄격한기준에서는0TP/3FP/F1 0이었고동일고정정상2/가스1도0TP/3FP였다.",
 "기본Bayes판정과안쪽학습기준은다른운용점이며좋은쪽만독립성능이라고주장하지않는다.",
 "같은605샷/시간블록20/purge10.정상592/가스13학습.비라벨은원본정제8007샷이고라벨을부여해학습한것이아니다.","",
 "## 기본Bayes판정",json.dumps(result,ensure_ascii=False,indent=2),"",
 "## 가스13개",p[p.actual.eq(1)][["TimeStamp","score","prediction_bayes"]].to_string(index=False),"",
 "비라벨은전체라벨재학습모델출력이다.비라벨정답없으므로위비율은실제불량률/정확도가아니다.",
 "새Run/미래독립검증이아니다.기존구간분리조건을유지한개발검증이며양성1~2개구간의한계는남는다."]
(ROOT/"reports/cn7_186_jiwoo_supervised_GMM_result.md").write_text("\n".join(lines),encoding="utf-8")


{
  "model": "GMM_N2_G1",
  "decision": "Bayes logodds>=0, no evaluation-label cutoff fit",
  "N": 605,
  "TP": 4,
  "FP": 12,
  "FN": 9,
  "TN": 580,
  "F1": 0.27586206896551724,
  "unlabeled_clean_shots": 8007,
  "unlabeled_alarms": 985,
  "unlabeled_alarm_percent": 12.301735981016611
}

ALL GAS
           TimeStamp      score  prediction_bayes
2020-10-16 05:21:38 -12.020532                 0
2020-10-16 05:37:29 -24.661419                 0
2020-10-16 05:45:26  14.951441                 1
2020-10-16 05:47:24 -14.207768                 0
2020-10-16 05:48:24 -12.995877                 0
2020-10-16 05:49:24  17.913410                 1
2020-10-16 05:50:25 -22.954514                 0
2020-10-16 05:51:23 -22.797976                 0
2020-10-16 05:52:23  -7.614139                 0
2020-10-16 05:53:23 -13.263393                 0
2020-10-16 05:54:21  -4.211633                 0
2020-10-16 05:56:22 452.397756                 1
2020-10-16 05:57:20 515.068105                 1

TRAIN CLASSES

1559